# 3. POS 데이터 이상치 전처리 (Final)

## 전처리 방침
1. `REFUND_NEGATIVE` 외의 모든 이상치 플래그 행 → **삭제**
   - 삭제 대상: `GIFT_OR_ZERO_QTY`, `ZERO_AMT_NEG_QTY`, `DATA_SUSPECT`, `PARTIAL_CANCEL`
   - 유지 대상: `NORMAL`, `REFUND_NEGATIVE`
2. `REFUND_NEGATIVE` 행 중 `상품명`에 **'공병'** 또는 **'수수료'** 포함 → **삭제**
3. 대분류가 **'공병공박스'** 인 행 → **삭제** (NORMAL 거래이지만 공병 반납으로 기록된 행, 1,976건)
4. 메모리 효율을 위해 **Parquet Row Group 단위 청크 처리** 후 저장
5. `flag_이상치_유형` 컬럼 제거 후 저장
6. 최종 결과를 `pos_data_전처리완료_final.parquet` 으로 저장

In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
import os

print(f"pandas   : {pd.__version__}")
print(f"pyarrow  : {pa.__version__}")

pandas   : 3.0.2
pyarrow  : 23.0.1


## 1. 파일 기본 정보 확인

In [4]:
INPUT_PATH  = 'C:/Users\송정현/Documents/Projects/박재홍교수님세미나/Projects/20기/7eleven_npd_framework/data/processed/최종/df_전처리완료.parquet'
OUTPUT_PATH = 'C:/Users\송정현/Documents/Projects/박재홍교수님세미나/Projects/20기/7eleven_npd_framework/data/processed/최종/pos_data_전처리완료_final.parquet'

pf = pq.ParquetFile(INPUT_PATH)
meta = pf.metadata
schema = pf.schema_arrow

print(f"총 Row Groups : {meta.num_row_groups}")
print(f"총 행 수      : {meta.num_rows:,}")
print(f"컬럼 수       : {meta.num_columns}")
print()
print("=== Schema ===")
print(schema)

총 Row Groups : 50
총 행 수      : 49,758,183
컬럼 수       : 20

=== Schema ===
영업일자: int64
판매시분초: int64
점포코드: int64
POS번호: int64
거래번호: int64
상품코드: large_string
매출수량: float
매출금액: float
상품명: large_string
상품대분류명: large_string
상품중분류명: large_string
상품소분류명: large_string
판매시간_dt: timestamp[us]
판매월: int8
판매주: int8
판매일: int8
판매시간대: int8
판매요일: dictionary<values=string, indices=int8, ordered=0>
거래_고유키: large_string
flag_이상치_유형: dictionary<values=string, indices=int8, ordered=0>
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 3153


## 2. 청크 단위 전처리 및 저장

Row Group 단위로 읽어 처리 후 바로 Parquet에 append 저장합니다.

## 2-1. 원거래 없는 환불 식별 (사전 처리)

REFUND_NEGATIVE 전체를 NORMAL과  매칭하여,
동일 점포·상품 기준으로 환불 시점 이전 정상거래가 없는 환불의 를 수집합니다.

In [5]:
import gc

print("원거래 없는 환불 식별 중...")

# REFUND 행 전체 로드 (약 34만건, 메모리 내 처리 가능)
df_refund_all = pd.read_parquet(
    INPUT_PATH,
    filters=[("flag_이상치_유형", "=", "REFUND_NEGATIVE")],
    columns=["거래_고유키", "점포코드", "상품코드", "판매시간_dt"]
)
df_refund_all["판매시간_dt"] = pd.to_datetime(df_refund_all["판매시간_dt"])
print(f"  REFUND 총 행수: {len(df_refund_all):,}")

# 환불 상품 목록으로 NORMAL 행 필터링 로드
refund_products = df_refund_all["상품코드"].unique().tolist()
df_normal_match = pd.read_parquet(
    INPUT_PATH,
    filters=[
        ("flag_이상치_유형", "=", "NORMAL"),
        ("상품코드", "in", refund_products)
    ],
    columns=["점포코드", "상품코드", "판매시간_dt"]
)
df_normal_match["판매시간_dt"] = pd.to_datetime(df_normal_match["판매시간_dt"])
print(f"  매칭 대상 NORMAL 행수: {len(df_normal_match):,}")

# merge_asof: 환불 시점 이전 동일 점포·상품의 정상거래 탐색
refund_sorted = df_refund_all.sort_values("판매시간_dt")
normal_sorted = df_normal_match.sort_values("판매시간_dt")

df_matched = pd.merge_asof(
    refund_sorted,
    normal_sorted[["점포코드", "상품코드", "판매시간_dt"]].rename(columns={"판매시간_dt": "_구매시간"}),
    left_on="판매시간_dt",
    right_on="_구매시간",
    by=["점포코드", "상품코드"],
    direction="backward"
)

# 매칭 성공 = _구매시간이 NaN이 아닌 환불
matched_refund_keys = set(df_matched.dropna(subset=["_구매시간"])["거래_고유키"])
unmatched_count = len(df_refund_all) - len(matched_refund_keys)

print(f"  매칭 성공: {len(matched_refund_keys):,}건")
print(f"  매칭 실패(제거 예정): {unmatched_count:,}건")

del df_refund_all, df_normal_match, df_matched, refund_sorted, normal_sorted
gc.collect()
print("사전 처리 완료")

원거래 없는 환불 식별 중...
  REFUND 총 행수: 1,029,631
  매칭 대상 NORMAL 행수: 47,774,978
  매칭 성공: 676,782건
  매칭 실패(제거 예정): 352,849건
사전 처리 완료


In [6]:
KEEP_FLAGS   = {'NORMAL', 'REFUND_NEGATIVE'}
REFUND_FLAG  = 'REFUND_NEGATIVE'
KEYWORD_PAT  = '공병|수수료'

total_in        = 0
removed_flag    = 0   # 비환불 이상치 삭제
removed_keyword   = 0   # 공병/수수료/공병공박스 삭제
removed_unmatched = 0   # 원거래 없는 환불 삭제
total_out       = 0

writer = None

for rg_idx in range(meta.num_row_groups):
    # Row Group 읽기
    chunk = pf.read_row_group(rg_idx).to_pandas()
    n_in = len(chunk)
    total_in += n_in

    # --- Step 1: NORMAL + REFUND_NEGATIVE 만 유지 ---
    mask_keep = chunk['flag_이상치_유형'].isin(KEEP_FLAGS)
    n_flag_removed = n_in - mask_keep.sum()
    removed_flag += n_flag_removed
    chunk = chunk[mask_keep]

    # --- Step 2: REFUND_NEGATIVE 중 공병/수수료 제거 + 공병공박스 대분류 제거 ---
    is_refund      = chunk['flag_이상치_유형'] == REFUND_FLAG
    has_kw         = chunk['상품명'].str.contains(KEYWORD_PAT, na=False)
    is_gonbyeong   = chunk['상품대분류명'] == '공병공박스'
    mask_remove    = (is_refund & has_kw) | is_gonbyeong

    # --- Step 3: 원거래 없는 환불 제거 ---
    is_unmatched_refund = is_refund & ~chunk['거래_고유키'].isin(matched_refund_keys)
    mask_remove = mask_remove | is_unmatched_refund

    n_kw_removed = (mask_remove & ~is_unmatched_refund).sum()
    n_unmatched  = is_unmatched_refund.sum()
    removed_keyword   += n_kw_removed
    removed_unmatched += n_unmatched
    chunk = chunk[~mask_remove]

    total_out += len(chunk)

    # --- flag_이상치_유형 컬럼 제거 ---
    chunk = chunk.drop(columns=['flag_이상치_유형'])

    # Parquet 스트리밍 저장
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
    writer.write_table(table)

    if (rg_idx + 1) % 5 == 0 or rg_idx == meta.num_row_groups - 1:
        print(f"  [RG {rg_idx+1:3d}/{meta.num_row_groups}] "
              f"누적 입력: {total_in:,}  /  누적 출력: {total_out:,}")

if writer:
    writer.close()

print()
print("=" * 55)
print("          전처리 결과 요약")
print("=" * 55)
print(f"  ▸ 원본 행 수               : {total_in:>12,} 행")
print(f"  ▸ Step1 삭제 (비환불 이상치): {removed_flag:>12,} 행")
print(f"  ▸ Step2 삭제 (공병/수수료/공병공박스): {removed_keyword:>12,} 행")
print(f"  ▸ Step3 삭제 (원거래없는환불)       : {removed_unmatched:>12,} 행")
print(f"  ▸ 총 삭제 행               : {removed_flag+removed_keyword+removed_unmatched:>12,} 행")
print(f"  ▸ 최종 행 수               : {total_out:>12,} 행")
print("=" * 55)

  [RG   5/50] 누적 입력: 5,048,576  /  누적 출력: 4,972,050
  [RG  10/50] 누적 입력: 10,000,000  /  누적 출력: 9,847,864
  [RG  15/50] 누적 입력: 15,048,576  /  누적 출력: 14,818,349
  [RG  20/50] 누적 입력: 20,000,000  /  누적 출력: 19,694,098
  [RG  25/50] 누적 입력: 25,048,576  /  누적 출력: 24,666,929
  [RG  30/50] 누적 입력: 30,000,000  /  누적 출력: 29,545,587
  [RG  35/50] 누적 입력: 35,048,576  /  누적 출력: 34,517,787
  [RG  40/50] 누적 입력: 40,000,000  /  누적 출력: 39,394,472
  [RG  45/50] 누적 입력: 45,048,576  /  누적 출력: 44,364,756
  [RG  50/50] 누적 입력: 49,758,183  /  누적 출력: 49,000,589

          전처리 결과 요약
  ▸ 원본 행 수               :   49,758,183 행
  ▸ Step1 삭제 (비환불 이상치):      717,247 행
  ▸ Step2 삭제 (공병/수수료/공병공박스):       24,625 행
  ▸ Step3 삭제 (원거래없는환불)       :       15,722 행
  ▸ 총 삭제 행               :      757,594 행
  ▸ 최종 행 수               :   49,000,589 행


## 3. 저장 결과 검증

In [7]:
df_check = pd.read_parquet(OUTPUT_PATH, columns=['매출수량', '매출금액', '상품명'])

print(f"[검증] 로드된 행 수: {len(df_check):,}")
print()
print("=== 저장된 컬럼 목록 ===")
print(pd.read_parquet(OUTPUT_PATH, columns=None).columns.tolist())
print()
print("=== 매출수량/매출금액 기초 통계 ===")
print(df_check[['매출수량', '매출금액']].describe())
print()

# 공병/수수료 잔존 여부 확인
kw_remain = df_check['상품명'].str.contains('공병|수수료', na=False).sum()
print(f"=== 공병/수수료 잔존 행수 (참고용): {kw_remain:,} ===")
print()
print(f"저장 경로: {os.path.abspath(OUTPUT_PATH)}")

[검증] 로드된 행 수: 49,000,589

=== 저장된 컬럼 목록 ===
['영업일자', '판매시분초', '점포코드', 'POS번호', '거래번호', '상품코드', '매출수량', '매출금액', '상품명', '상품대분류명', '상품중분류명', '상품소분류명', '판매시간_dt', '판매월', '판매주', '판매일', '판매시간대', '판매요일', '거래_고유키']

=== 매출수량/매출금액 기초 통계 ===
               매출수량          매출금액
count  4.900059e+07  4.900059e+07
mean   1.328053e+00  3.317169e+03
std    2.943018e+00  3.716046e+05
min   -2.000000e+03 -1.300000e+09
25%    1.000000e+00  1.500000e+03
50%    1.000000e+00  2.300000e+03
75%    1.000000e+00  4.500000e+03
max    2.000000e+03  1.300000e+09

=== 공병/수수료 잔존 행수 (참고용): 322,184 ===

저장 경로: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\최종\pos_data_전처리완료_final.parquet
